In [21]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import sklearn

In [22]:
import os
import sys

print(sys.executable)
print(os.getpid())


d:\Chest_XRay_Project\chest_env\Scripts\python.exe
9924


In [23]:
import sys
import torch

print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.__file__)

d:\Chest_XRay_Project\chest_env\Scripts\python.exe
2.13.0+cu126
12.6
True
d:\Chest_XRay_Project\chest_env\Lib\site-packages\torch\__init__.py


In [24]:
print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 4060 Laptop GPU


In [25]:
from PIL import Image
from torchvision import transforms

In [26]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

In [27]:
train_val_list = pd.read_csv('D:/NIH chest X-Ray/train_val_list.txt',
                             header = None,
                             names = ["Image Index"])

test_list = pd.read_csv("D:/NIH chest X-Ray/test_list.txt",
                        header = None,
                        names = ["Image Index"])

In [28]:
train_val_list

,Image Index
0,00000001_000.png
1,00000001_001.png
2,00000001_002.png
3,00000002_000.png
4,00000004_000.png
...,...
86519,00030789_000.png
86520,00030793_000.png
86521,00030795_000.png
86522,00030801_000.png


In [29]:
test_list

,Image Index
0,00000003_000.png
1,00000003_001.png
2,00000003_002.png
3,00000003_003.png
4,00000003_004.png
...,...
25591,00030800_000.png
25592,00030802_000.png
25593,00030803_000.png
25594,00030804_000.png


In [30]:
import ast

df3=pd.read_csv('D:/Chest_XRay_Project/notebooks/result.csv')
df3["Labels_encoded"] = df3["Labels_encoded"].apply(ast.literal_eval)


In [31]:
train_val_df = df3[
    df3["Image Index"].isin(train_val_list["Image Index"])
]

test_df = df3[
    df3["Image Index"].isin(test_list["Image Index"])
]

In [32]:
patients = train_val_df["Patient ID"].unique()

print(len(patients))

28008


In [33]:
from sklearn.model_selection import train_test_split

train_patients, val_patients = train_test_split(
    patients,
    test_size=0.1,
    random_state=42
)

In [34]:
train_df = train_val_df[
    train_val_df["Patient ID"].isin(train_patients)
]

val_df = train_val_df[
    train_val_df["Patient ID"].isin(val_patients)
]

In [35]:
train_df.to_csv('train.csv', index=False)
val_df.to_csv('validation.csv', index=False)

In [36]:
train_df

,Image Index,Image Path,Finding Labels,Patient ID,Labels_encoded
0,00000001_000.png,D:\NIH chest X-Ray\images_001\images\00000001_...,['Cardiomegaly'],1,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,00000001_001.png,D:\NIH chest X-Ray\images_001\images\00000001_...,"['Cardiomegaly', 'Emphysema']",1,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,00000001_002.png,D:\NIH chest X-Ray\images_001\images\00000001_...,"['Cardiomegaly', 'Effusion']",1,"[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,00000002_000.png,D:\NIH chest X-Ray\images_001\images\00000002_...,['No Finding'],2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
12,00000004_000.png,D:\NIH chest X-Ray\images_001\images\00000004_...,"['Mass', 'Nodule']",4,"[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0]"
...,...,...,...,...,...
112100,00030789_000.png,D:\NIH chest X-Ray\images_012\images\00030789_...,['Infiltration'],30789,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
112106,00030793_000.png,D:\NIH chest X-Ray\images_012\images\00030793_...,"['Mass', 'Nodule']",30793,"[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0]"
112108,00030795_000.png,D:\NIH chest X-Ray\images_012\images\00030795_...,['Pleural_Thickening'],30795,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]"
112114,00030801_000.png,D:\NIH chest X-Ray\images_012\images\00030801_...,['No Finding'],30801,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [37]:
from torch.utils.data import Dataset



class XrayDataset(Dataset):
    def __init__(self,df,transform=None):
        self.df = df.reset_index(drop = True)
        self.transform = transform 
        
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        
        img_path = row["Image Path"]
        label = row["Labels_encoded"]
        
        image = Image.open(img_path)
        
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.float32)
            
        return image, label

In [38]:
from torch.utils.data import DataLoader

train_dataset = XrayDataset(train_df, transform = transform)

val_dataset = XrayDataset(val_df, transform = transform)

test_dataset = XrayDataset(test_df, transform = transform)

train_dataloader = DataLoader(train_dataset, batch_size = 32, shuffle = True, num_workers = 0, pin_memory=True )

test_dataloader = DataLoader(test_dataset, batch_size = 32, shuffle = True, num_workers = 0, pin_memory=True )

In [39]:
#images, labels = next(iter(train_dataloader))
#print(images.shape) 
#print(labels.shape) 

In [40]:

import torch.nn as nn
import torchvision.models
from torchvision.models import densenet121

device = torch.device("cuda")

model = densenet121(weights="DEFAULT")

NUM_CLASSES = 14

model.classifier = nn.Linear(
    model.classifier.in_features,
    NUM_CLASSES
)

model = model.to(device)



In [41]:
criterion = nn.BCEWithLogitsLoss()

In [42]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 1e-4
)

In [43]:
print(next(model.parameters()).device)

cuda:0


In [44]:
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.13.0+cu126
CUDA version: 12.6
CUDA available: True


In [45]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [46]:
print(torch.version.cuda)

12.6


In [47]:
import sys
print(sys.executable)

d:\Chest_XRay_Project\chest_env\Scripts\python.exe


In [48]:
print(next(model.parameters()).device)

cuda:0


In [49]:
print(torch.cuda.memory_allocated()/1024**2)

27.00390625


In [50]:
import time

start = time.time()

image, label = train_dataset[0]

print("Single image time:", time.time()-start)
print(image.shape)
print(label.shape)

Single image time: 0.03468036651611328
torch.Size([3, 224, 224])
torch.Size([14])


In [51]:
len(train_dataset)

77988

In [52]:
import os
num_epochs=15


os.makedirs("D:/Chest_XRay_Project/models/training", exist_ok=True)

for epoch in range(num_epochs):
    model.train()
    
    epoch_loss = 0
    
    for images,labels in train_dataloader:
        
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        optimizer.step()
        
        torch.cuda.synchronize()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_dataloader)
    
    print(f"Epoch {epoch+1} completed")
    
    print(f"Epoch {epoch+1}/{num_epochs} Loss: {avg_loss:.4f}")
          
    

    torch.save(
        model.state_dict(),
        f"D:/Chest_XRay_Project/models/training/densenet121_epoch_new_{epoch+1}.pth"
    )        
        
        
        
        

Epoch 1 completed
Epoch 1/15 Loss: 0.1455
Epoch 2 completed
Epoch 2/15 Loss: 0.1301
Epoch 3 completed
Epoch 3/15 Loss: 0.1240
Epoch 4 completed
Epoch 4/15 Loss: 0.1164
Epoch 5 completed
Epoch 5/15 Loss: 0.1061
Epoch 6 completed
Epoch 6/15 Loss: 0.0920
Epoch 7 completed
Epoch 7/15 Loss: 0.0750
Epoch 8 completed
Epoch 8/15 Loss: 0.0573
Epoch 9 completed
Epoch 9/15 Loss: 0.0432
Epoch 10 completed
Epoch 10/15 Loss: 0.0337
Epoch 11 completed
Epoch 11/15 Loss: 0.0286
Epoch 12 completed
Epoch 12/15 Loss: 0.0245
Epoch 13 completed
Epoch 13/15 Loss: 0.0221
Epoch 14 completed
Epoch 14/15 Loss: 0.0202
Epoch 15 completed
Epoch 15/15 Loss: 0.0192


In [53]:
#torch.save(    model.state_dict(),    "D:/Chest_XRay_Project/models/densenet121_epoch1.pth")